# GP Diagnostic: Does Red Noise Shift the Likelihood Peak?

## Background

In `01b_2CW_mode_breaking.ipynb` a 1D distance scan over the widest-prior pulsar
(B1953+29) with a noise-free 1-CW injection does **not** peak at the truth distance
(ΔlnL ≈ −4.8).

## Suspicion

`build_noisefree_likelihood` calls `ds.makecommongp_fourier(..., name="rednoise")`.
This creates a **common** (globally-named) red-noise GP, whose parameters are likely
`rednoise_log10_A` and `rednoise_gamma`.

But the code then sets **per-pulsar** names in `param_dict`:
```python
param_dict[f"{psr.name}_rednoise_log10_A"] = -20.0
param_dict[f"{psr.name}_rednoise_gamma"]   =  4.0
```

If these per-pulsar keys don't match what `logl.params` expects, the
filter `{k: v ... if k in logl.params}` silently drops them.
The global `rednoise_log10_A` is then absent from `base_values_a`
and discovery defaults to a non-negligible amplitude, causing the GP to absorb
the CW signal and shift the peak.

## Plan

| Cell | What it does |
|------|-------------|
| Setup | Inject CW_A, build original likelihood |
| Inspect params | Print all `logl.params`, flag missing rednoise/gwb entries |
| Pure-CW likelihood | Rebuild with `commongp=None, globalgp=None` |
| 1D scan | Compare original vs pure-CW scan for B1953+29 |
| Conclusion | Verdict + fix recipe |

In [ ]:
import sys, importlib
sys.path.insert(0, "../CW_lnL_check")

import numpy as np
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import time

import discovery as ds
import cw_helpers as cwh
importlib.reload(cwh)
from cw_helpers import (
    load_pulsars, inject_noisefree_cw, build_noisefree_likelihood,
    scan_pulsar_distance, analyze_peaks, compute_mode_spacing, MultiSourceDelay,
)

## Step 1 — Setup: inject CW_A, build original likelihood

In [ ]:
N_PSR = 5
ent_psrs, disco_psrs = load_pulsars(N_PSR)
sorted_psrs = sorted(disco_psrs, key=lambda p: p.pdist[1])
target_psr = sorted_psrs[-1]

print(f"Target (widest prior): {target_psr.name}")
print(f"  mu={target_psr.pdist[0]:.4f} kpc   sigma={target_psr.pdist[1]:.4f} kpc")

# Single CW injection — same as 01b
CW_A = [{"cos_gwtheta": 0.3, "gwphi": 2.5, "cos_inc": -0.2,
          "log10_mc": 9.0, "log10_fgw": -8.0, "log10_h": -12.0,
          "phase0": 1.0, "psi": 0.7}]

t0 = time.time()
residual_map_a, _ = inject_noisefree_cw(disco_psrs, CW_A)
logl_fn_a, param_keys_a, base_values_a = build_noisefree_likelihood(
    disco_psrs, residual_map_a, num_cw=1, cw_params_list=CW_A
)
_ = logl_fn_a(base_values_a)  # warm JIT
print(f"Original likelihood built + JIT warmed: {time.time()-t0:.0f} s")
print(f"lnL(truth) = {float(logl_fn_a(base_values_a)):.4f}")
print(f"Number of params in base_values_a: {len(param_keys_a)}")

## Step 2A — Inspect `param_keys_a`

Print every parameter in the original likelihood vector, then highlight
which rednoise/gwb names made it in and which were silently dropped.

In [ ]:
# ── What does logl.params contain? ───────────────────────────────────────────
# Rebuild the fml object to access logl.params directly.
# (logl_fn_a is JIT-wrapped; param_keys_a is already the filtered list.)

print("=== All parameter names in param_keys_a ===")
for k, v in zip(param_keys_a, base_values_a):
    print(f"  {k:55s}  {float(v):10.4f}")

print()
print("=== rednoise / gwb params in param_keys_a ===")
found_rn_gwb = [(k, float(v)) for k, v in zip(param_keys_a, base_values_a)
                if "rednoise" in k or "gwb" in k]
if found_rn_gwb:
    for k, v in found_rn_gwb:
        print(f"  {k:55s}  {v:10.4f}")
else:
    print("  NONE — these parameters were NOT included in the likelihood!")

print()
print("=== params injected into param_dict but absent from param_keys_a ===")
# Reconstruct what build_noisefree_likelihood put in param_dict
attempted = {}
for psr in disco_psrs:
    attempted[f"{psr.name}_rednoise_log10_A"] = -20.0
    attempted[f"{psr.name}_rednoise_gamma"]   =   4.0
attempted["gwb_log10_A"] = -20.0
attempted["gwb_gamma"]   =  4.333
missing = [k for k in attempted if k not in param_keys_a]
if missing:
    for k in missing:
        print(f"  MISSING: {k}  (tried to set to {attempted[k]})")
else:
    print("  None — all param_dict entries made it into param_keys_a")

## Step 2B — Rebuild `ArrayLikelihood` to inspect `logl.params` directly

This reveals the **actual parameter names** that discovery's common GP expects.
Compare against what `build_noisefree_likelihood` tried to set.

In [ ]:
# ── What does the discovery likelihood actually expect? ───────────────────────
# Rebuild the ArrayLikelihood so we can inspect logl.params directly.

noisedict = {psr.name + "_KAT_MKBF_efac": 1.0 for psr in disco_psrs}
noisedict.update({psr.name + "_KAT_MKBF_log10_ecorr": -8.0 for psr in disco_psrs})
noisedict.update({psr.name + "_KAT_MKBF_log10_t2equad": -8.0 for psr in disco_psrs})

noise_terms   = {psr.name: ds.makenoise_measurement(psr, noisedict=noisedict) for psr in disco_psrs}
timing_terms  = {psr.name: ds.makegp_timing(psr, variance=1e-14) for psr in disco_psrs}
T             = ds.getspan(disco_psrs)
common_gp     = ds.makecommongp_fourier(disco_psrs, ds.powerlaw, 30, T, name="rednoise")
global_gp     = ds.makeglobalgp_fourier(disco_psrs, ds.powerlaw, ds.hd_orf, 30, T, name="gwb")

pulsar_likes = [
    ds.PulsarLikelihood([
        np.array(residual_map_a[psr.name], copy=True),
        noise_terms[psr.name],
        timing_terms[psr.name],
        MultiSourceDelay(psr, 1, include_pterm=True),
    ])
    for psr in disco_psrs
]
fml_diag = ds.ArrayLikelihood(pulsar_likes, commongp=common_gp, globalgp=global_gp)
logl_diag = fml_diag.logL

print(f"All {len(logl_diag.params)} params expected by logl:")
for k in logl_diag.params:
    in_pk = "(in param_keys_a)" if k in param_keys_a else "*** NOT IN param_keys_a ***"
    print(f"  {k:55s}  {in_pk}")

print()
all_rn = [k for k in logl_diag.params if "rednoise" in k]
all_gwb = [k for k in logl_diag.params if "gwb" in k]
print(f"rednoise params expected: {all_rn}")
print(f"gwb params expected:      {all_gwb}")

rn_provided = [k for k in all_rn  if k in param_keys_a]
gwb_provided= [k for k in all_gwb if k in param_keys_a]
print()
print(f"rednoise actually provided in base_values_a: {rn_provided}")
print(f"gwb     actually provided in base_values_a: {gwb_provided}")

## Step 3 — `build_pure_cw_likelihood`: no GP terms

White noise + timing model + CW only. No common red noise, no GWB.
If the scan peaks at truth here but not with the original, the GP is the culprit.

In [ ]:
def build_pure_cw_likelihood(disco_psrs, residual_map, cw_params_list,
                              log10_equad=-8.0):
    """White noise + timing model + CW only — no GP terms.

    Using ds.ArrayLikelihood(commongp=None) crashes in JIT because PulsarLikelihood.logL
    calls make_kernelproduct(y_eval) inside JIT with a traced (non-callable) y, which
    hits the scipy path and raises TracerArrayConversionError.

    Fix: directly build psl.N.make_kernelproduct(psl.y) with the callable CompoundDelay.
    WoodburyKernel_novar routes callable y through jsp (JAX scipy) inside the returned
    function — JIT-safe. No commongp overhead at all.
    """
    num_cw = len(cw_params_list)

    noisedict = {}
    for psr in disco_psrs:
        noisedict[psr.name + "_KAT_MKBF_efac"] = 1.0
        noisedict[psr.name + "_KAT_MKBF_log10_ecorr"] = -8.0
        noisedict[psr.name + "_KAT_MKBF_log10_t2equad"] = log10_equad

    pulsar_likes = [
        ds.PulsarLikelihood([
            np.array(residual_map[psr.name], copy=True),
            ds.makenoise_measurement(psr, noisedict=noisedict),
            ds.makegp_timing(psr, variance=1e-14),
            MultiSourceDelay(psr, num_cw, include_pterm=True),
        ])
        for psr in disco_psrs
    ]

    # psl.N  = WoodburyKernel_novar  (white noise + timing GP, all constant)
    # psl.y  = CompoundDelay          (callable: residuals - CW_delay(params))
    # WoodburyKernel_novar.make_kernelproduct(callable) uses jsp inside => JIT-safe.
    kp_fns = [psl.N.make_kernelproduct(psl.y) for psl in pulsar_likes]
    all_params = sorted(set.union(*[set(kp.params) for kp in kp_fns]))

    disco_suffixes = ["" if i == 0 else f"_{i+1}" for i in range(num_cw)]
    param_dict = {}
    for suffix, cw_p in zip(disco_suffixes, cw_params_list):
        for key in MultiSourceDelay.global_params:
            param_dict[f"cw_{key}{suffix}"] = cw_p[key]
    for psr in disco_psrs:
        param_dict[f"{psr.name}_cw_p_dist"] = psr.pdist[0]

    order_map = {k: i for i, k in enumerate(all_params)}
    sorted_items = sorted(
        param_dict.items(), key=lambda kv: order_map.get(kv[0], float("inf"))
    )
    sorted_dict = {k: v for k, v in sorted_items if k in all_params}
    param_keys = list(sorted_dict.keys())
    base_vals = jnp.array([sorted_dict[k] for k in param_keys], dtype=jnp.float64)

    def logl_wrapped(x_array):
        params = {k: v for k, v in zip(param_keys, x_array)}
        return sum(kp(params) for kp in kp_fns)

    return jax.jit(logl_wrapped), param_keys, base_vals


In [ ]:
t0 = time.time()
logl_fn_pure, param_keys_pure, base_values_pure = build_pure_cw_likelihood(
    disco_psrs, residual_map_a, CW_A
)
_ = logl_fn_pure(base_values_pure)  # warm JIT
print(f"Pure-CW likelihood built + JIT warmed: {time.time()-t0:.0f} s")

print(f"\nPure-CW params ({len(param_keys_pure)} total):")
for k, v in zip(param_keys_pure, base_values_pure):
    print(f"  {k:50s}  {float(v):.6f}")

print(f"\nlnL_pure(truth) = {float(logl_fn_pure(base_values_pure)):.4f}")
print(f"lnL_orig(truth) = {float(logl_fn_a(base_values_a)):.4f}")

## Step 4 — 1D scan: original vs pure-CW likelihood

In [ ]:
mu, sig = target_psr.pdist
scan_min = max(0.01, mu - 3.0 * sig)
scan_max = mu + 3.0 * sig
N_POINTS = 2000
dist_key = f"{target_psr.name}_cw_p_dist"

dL = compute_mode_spacing(
    CW_A[0]["cos_gwtheta"], CW_A[0]["gwphi"], CW_A[0]["log10_fgw"], target_psr.pos
)
print(f"Mode spacing for {target_psr.name}: dL = {dL:.4f} kpc")
print(f"Scanning [{scan_min:.3f}, {scan_max:.3f}] kpc  ({N_POINTS} pts)")

t0 = time.time()
scan_d_a, scan_ll_a = scan_pulsar_distance(
    logl_fn_a, base_values_a, param_keys_a, dist_key, scan_min, scan_max, N_POINTS, required_points=[mu]
)
ta = time.time() - t0
print(f"  original: {ta:.1f} s")

scan_d_p, scan_ll_p = scan_pulsar_distance(
    logl_fn_pure, base_values_pure, param_keys_pure, dist_key, scan_min, scan_max, N_POINTS, required_points=[mu]
)
print(f"  pure-CW:  {time.time()-t0-ta:.1f} s")

# Truth lnL and peak for each
truth_ll_a = float(np.interp(mu, scan_d_a, scan_ll_a))
truth_ll_p = float(np.interp(mu, scan_d_p, scan_ll_p))
max_ll_a, max_ll_p = np.max(scan_ll_a), np.max(scan_ll_p)
max_d_a = float(scan_d_a[np.argmax(scan_ll_a)])
max_d_p = float(scan_d_p[np.argmax(scan_ll_p)])

print()
print(f"{'':25s}  {'orig':>14}  {'pure-CW':>14}")
print("-" * 57)
print(f"{'truth dist (kpc)':25s}  {mu:14.4f}  {mu:14.4f}")
print(f"{'grid-max dist (kpc)':25s}  {max_d_a:14.4f}  {max_d_p:14.4f}")
print(f"{'lnL(truth)':25s}  {truth_ll_a:14.4f}  {truth_ll_p:14.4f}")
print(f"{'lnL(grid max)':25s}  {max_ll_a:14.4f}  {max_ll_p:14.4f}")
print(f"{'delta lnL (truth - max)':25s}  {truth_ll_a-max_ll_a:14.4f}  {truth_ll_p-max_ll_p:14.4f}")
print()
if abs(truth_ll_p - max_ll_p) < 1.0:
    print("RESULT: pure-CW likelihood peaks at truth => GP terms are the problem.")
elif abs(truth_ll_a - max_ll_a) < 1.0:
    print("RESULT: original peaks at truth, pure does not — unexpected; investigate.")
else:
    print("RESULT: BOTH likelihoods offset from truth => deeper signal model mismatch.")


## Step 5 — Plot and verdict

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle(
    f"GP diagnostic: 1D scan — {target_psr.name}\n"
    f"truth = {mu:.4f} kpc   dL = {dL:.4f} kpc",
    fontsize=12,
)
clip_min = -50.0

for ax, (label, sd, sl, max_d, delta) in zip(axes, [
    ("original (with GP)", scan_d_a, scan_ll_a, max_d_a, truth_ll_a - max_ll_a),
    ("pure CW (no GP)",   scan_d_p, scan_ll_p, max_d_p, truth_ll_p - max_ll_p),
]):
    sl_norm = sl - np.max(sl)
    ax.plot(sd, np.clip(sl_norm, clip_min, 0), "C0", lw=0.8)

    # Prior shading
    ax.axvspan(max(0.01, mu - 3*sig), mu + 3*sig, alpha=0.07, color="gray",
               label="+-3 sigma prior")
    ax.axvspan(max(0.01, mu - sig),   mu + sig,   alpha=0.10, color="gray")

    # Truth
    ax.axvline(mu, color="red", lw=2.0, ls="--", label=f"truth = {mu:.4f}")
    truth_norm = float(np.interp(mu, sd, sl_norm))
    ax.plot(mu, np.clip(truth_norm, clip_min, 0), "r*", ms=12, zorder=5)

    # Grid max
    if abs(max_d - mu) > 0.01 * dL:
        ax.axvline(max_d, color="orange", lw=1.5, ls=":", label=f"grid max = {max_d:.4f}")

    ax.text(0.97, 0.97,
            f"delta lnL = {delta:+.3f}\n"
            f"(truth - global max)\n"
            f"grid max at {max_d:.4f} kpc",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=9, family="monospace",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.85))

    ax.set_xlabel("Distance (kpc)", fontsize=11)
    ax.set_ylabel("Delta ln L  (max = 0)", fontsize=11)
    ax.set_title(label, fontsize=12)
    ax.set_ylim(clip_min * 1.05, 2.0)
    ax.legend(fontsize=9, loc="lower left")
    ax.grid(True, alpha=0.3, lw=0.5)

plt.tight_layout()
plt.savefig("01b_gp_diagnostic.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: 01b_gp_diagnostic.pdf")

In [ ]:
# ── Conclusion and next steps ─────────────────────────────────────────────────
delta_orig = truth_ll_a - max_ll_a
delta_pure = truth_ll_p - max_ll_p

print("=" * 60)
print("DIAGNOSTIC SUMMARY")
print("=" * 60)
print(f"  Original likelihood  delta_lnL = {delta_orig:+.3f}")
print(f"  Pure-CW likelihood   delta_lnL = {delta_pure:+.3f}")
print()

if delta_pure > -1.0 and delta_orig < -1.0:
    print("VERDICT: GP terms shift the peak away from truth.")
    print()
    print("Root cause (check cell above):")
    missing_rn = [k for k in logl_diag.params if "rednoise" in k and k not in param_keys_a]
    missing_gwb= [k for k in logl_diag.params if "gwb"      in k and k not in param_keys_a]
    if missing_rn or missing_gwb:
        print("  makecommongp_fourier uses GLOBAL param names (no pulsar prefix).")
        print("  build_noisefree_likelihood sets PER-PULSAR names => mismatch.")
        print("  The global rednoise_log10_A is NOT set in base_values_a.")
        print("  Discovery uses its internal default (~-14) => GP absorbs CW signal.")
        print()
        print("FIX OPTIONS:")
        print("  1. Use build_pure_cw_likelihood for all noise-free distance scans.")
        print("  2. Or set global param names in build_noisefree_likelihood:")
        for k in missing_rn:
            print(f"       param_dict['{k}'] = -20.0")
        for k in missing_gwb:
            print(f"       param_dict['{k}'] = -20.0  (or 4.333 for gamma)")
    else:
        print("  All params are present — investigate further.")
elif delta_pure < -1.0 and delta_orig < -1.0:
    print("VERDICT: BOTH likelihoods offset from truth.")
    print("GP terms are NOT the problem.")
    print("Possible causes:")
    print("  - evolve=True in enterprise but discovery uses non-evolving template")
    print("  - reference time (tref) mismatch between injection and likelihood")
    print("  - unit error in pulsar distance (kpc vs pc) or toa (s vs us)")
    print("  - wrong pulsar ordering (ent_psrs[i] != disco_psrs[i])")
elif delta_pure > -1.0 and delta_orig > -1.0:
    print("VERDICT: Both peak at truth — no problem detected.")
    print("The reported ΔlnL ≈ -4.8 was not reproduced here.")
else:
    print("VERDICT: Ambiguous. Inspect plots and cell outputs above.")